In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import datetime

In [2]:
# !pip install psycopg2
# http://www.pygresql.org/download/index.html
import psycopg2

### ЗАДАНИЕ

Необходимо: 
1. Написать SQL-запрос, который будет рассчитывать среднее время ответа для каждого менеджера/пары менеджеров. 
Расчёт должен учитывать следующее: 
• если в диалоге идут несколько сообщений подряд от клиента или менеджера, то при расчёте времени ответа надо учитывать только первое сообщение из каждого блока; 
• менеджеры работают с 09:30 до 00:00, поэтому нерабочее время не должно учитываться в расчёте среднего времени ответа, т.е. если клиент написал в 23:59, а менеджер ответил в 09:30 – время ответа равно одной минуте; 
• ответы на сообщения, пришедшие ночью также нужно учитывать. 
2. На основе базы данных из первого задания построить дашборд в DataLens с данными о среднем времени ответа менеджеров. Виды визуализаций и структура отчёта произвольные, однако необходима возможность фильтровать данные по дням, менеджерам и начальникам отделов продаж. 
3. Решить первое задание при помощи Python и библиотеки pandas


# 2 задание.

Сслыка на дашборд DataLens по адресу:

https://datalens.yandex/3hw5o3wqdgcwq


___________________________

# 3 задание.

Исследование данных с помощью Python Pandas.

In [3]:
# Подсоединились ------------------------------------------------------------------------------------
db = psycopg2.connect("""
    host=rc1a-p8bp15mmxsfwpbt0.mdb.yandexcloud.net
    port=6432
    dbname=db1
    user=test_user
    password=j2M{CnnFq@
    """)

# cur = db.cursor()
# cur.execute('SELECT version()')
# print(cur.fetchone()[0])
# ----------------------------

In [4]:
# Скачиваем необходимые данные
df_man = pd.read_sql_query(
    "SELECT * FROM test.managers",
    db)
df_mes = pd.read_sql_query(
    "SELECT * FROM test.chat_messages",
    db)
df_rops = pd.read_sql_query(
    "SELECT * FROM test.rops",
    db)

In [5]:
# Закрываем соединение
db.close()

In [6]:
df_man.head()

,mop_id,name_mop,rop_id
0,6645315,Гюнель и Илина,1
1,6744792,Юля и Наташа,1
2,10262505,Вика и Марго,1
3,10262513,Ира и Варя,1
4,10465254,Настя и Даша,1


In [7]:
df_mes.head()

,message_id,type,entity_id,created_by,created_at
0,"""01jb7da570sf4f65xdf0ptvv9q""",incoming_chat_message,37556493,0,1730046924
1,"""01jb3t6bc8gvyfnd063nd91ng4""",incoming_chat_message,37549491,0,1729926213
2,"""01jb6gj9ngwk0ybbmg9w90pbqg""",incoming_chat_message,37531455,0,1730016782
3,"""01jb75bkprk0d7hht1g5vy1pm4""",incoming_chat_message,37553371,0,1730038583
4,"""01jb3t5xprbph1c6pym8ervxpm""",incoming_chat_message,37548675,0,1729926199


In [8]:
df_rops.head()

,rop_id,rop_name
0,1,Катя РОП
1,2,Полина РОП
2,3,Эля РОП


In [9]:
# Объединяем с менеджерами. Создаем датасет
df = pd.merge(df_mes.rename(columns={'created_by': 'mop_id'}), df_man, on='mop_id', how='left')

In [10]:
# Смотрим пустые ячейки
df.isna().sum()

message_id        0
type              0
entity_id         0
mop_id            0
created_at        0
name_mop      10313
rop_id        10313
dtype: int64

In [11]:
# Заполняем пустые ячейки
df.fillna(0, inplace=True)

In [12]:
# Переводим в необходимый формат
df['rop_id'] = df['rop_id'].astype('int')
# Объединяем
df = pd.merge(df, df_rops, on='rop_id', how='left')
# Заполняем пустые
df.fillna(0, inplace=True)

In [14]:
# Переводим из unix в datetime
df['created_at_datetime'] = pd.to_datetime(df['created_at'],unit='s')
# Сортируем для удобства работы с датасетом
df.sort_values(by=['entity_id', 'created_at_datetime'], inplace=True)

In [15]:
df

,message_id,type,entity_id,mop_id,created_at,name_mop,rop_id,rop_name,created_at_datetime
3107,"""01jb4pjb1g22ts7z6arcjv11w6""",incoming_chat_message,35810509,0,1729955966,0,0,0,2024-10-26 15:19:26
4102,"""01jb4pncn7gm1re1zdxkzhnp6s""",outgoing_chat_message,35810509,10262493,1729956065,Ками и Мила,3,Эля РОП,2024-10-26 15:21:05
6430,"""01jb4qedfgxnd2yqr4w26hs4q5""",incoming_chat_message,35810509,0,1729956886,0,0,0,2024-10-26 15:34:46
6639,"""01jb4qj2skc2wjfdm8qaf69yjh""",outgoing_chat_message,35810509,10262493,1729957006,Ками и Мила,3,Эля РОП,2024-10-26 15:36:46
17586,"""01jb7gxwb0sepv9vtekrgj3v6d""",incoming_chat_message,35945535,0,1730050716,0,0,0,2024-10-27 17:38:36
...,...,...,...,...,...,...,...,...,...
18914,"""01jb7w5npw67ghth0ctgqkvrzr""",outgoing_chat_message,37557215,6744792,1730062505,Юля и Наташа,1,Катя РОП,2024-10-27 20:55:05
18864,"""01jb7vxxz0djjdc4qjezwhycev""",incoming_chat_message,37557225,0,1730062252,0,0,0,2024-10-27 20:50:52
18908,"""01jb7w3wcr7w96me3k4t5y9a3w""",incoming_chat_message,37557229,0,1730062447,0,0,0,2024-10-27 20:54:07
18925,"""01jb7we9ap70azqhbmqsvxqafh""",outgoing_chat_message,37557233,6780177,1730062787,Даша и Даша,2,Полина РОП,2024-10-27 20:59:47


In [16]:
# Делаем смещение в каждом блоке, чтобы затем исключить несколько сообщений подряд
for num in tqdm(df['entity_id'].unique().tolist()):    
    df.loc[df['entity_id'] == num, "name_mop2"] = df.loc[df['entity_id'] == num]['name_mop'].shift(1)

100%|██████████| 3544/3544 [00:02<00:00, 1425.39it/s]


In [17]:
# Смотрим сообщения, в которых есть ошибки или пишет 1 пользователь/менеджер
df_tmp = df.groupby(['entity_id'], as_index=False).agg({'mop_id': 'nunique'})
df_tmp.loc[df_tmp['mop_id'] < 2]

,entity_id,mop_id
3,36209333,1
4,36222379,1
5,36308085,1
7,36341869,1
8,36358037,1
...,...,...
3538,37557207,1
3540,37557225,1
3541,37557229,1
3542,37557233,1


In [18]:
# Убираем из датасета одиночные события
df = df.loc[~df['entity_id'].isin(df_tmp.loc[df_tmp['mop_id'] < 2]['entity_id'])]

In [19]:
# Убираем из датасета несколько сообщений подряд от одной из сторон переписки (оставляем только связку вопрос-ответ)
df = df.loc[~df.index.isin(df.loc[df['name_mop'] == df['name_mop2']].index.tolist())]

In [20]:
# Делаем смещение по дате в каждом блоке entity_id, чтобы потом рассчитать время ответа
for num in tqdm(df['entity_id'].unique().tolist()):
    df.loc[df['entity_id'] == num, "created_at_datetime2"] = df.loc[df['entity_id'] == num]['created_at_datetime'].shift(1)
    

100%|██████████| 1581/1581 [00:01<00:00, 1394.09it/s]


In [21]:
# Делаем расчет времени ответа
df['created_at_diff'] = df['created_at_datetime'] - df['created_at_datetime2']
# Переводим в секунды
df['created_at_diff_sec'] = df['created_at_diff'].dt.total_seconds()
# Берем только время без даты
df['created_at_time'] = pd.to_datetime(df['created_at_datetime'], format='%H:%M').dt.time
# Рассчитываем нерабочее время
df['not_work_time'] = df['created_at_time'] < datetime.time(9, 30, 0)

In [22]:
# Рассчитываем время в секундах, которое нужно вычесть из времени ответа менеджера, если оно на стыке рабоче и нерабочего времени
df['minus_time'] = 0
df.loc[df['not_work_time'] == True, 'minus_time'] = (pd.to_datetime('2000-01-01' + " " + df['created_at_time'].astype('str')) - pd.to_datetime(f"2000-01-01 09:30:00")).dt.total_seconds()

In [23]:
# Заполняем пустоты
df['minus_time'].fillna(0, inplace=True)
# Вычитаем из расчетного времени то время ответа, которое на стыке рабочего и нерабочего времени
df['created_at_diff_sec'] = df['created_at_diff_sec'] - np.abs(df['minus_time'])
# Заменяем отрицательные значения нулями(вопрос-ответ был строго в нерабочее время)
df.loc[(df['not_work_time'] == True) & (df['created_at_diff_sec'] < 0), 'created_at_diff_sec'] = 0

In [25]:
# Делаем финальную группировку и вычисления

final_df = pd.concat([df.loc[(~df['created_at_diff_sec'].isna()) & (~df['not_work_time'] == True)].groupby(
        'name_mop', as_index=False).agg({'created_at_diff_sec': 'mean'}), 
    df.loc[(~df['created_at_diff_sec'].isna()) & (~df['not_work_time'] == True)].groupby(
        'name_mop', as_index=False).agg({'created_at_diff_sec': 'median'})[['created_at_diff_sec']].rename(
            columns={'created_at_diff_sec': 'created_at_diff_sec_median'})
                         ], axis=1)

final_df['created_at_diff_minutes'] = final_df['created_at_diff_sec'] / 60
final_df['created_at_diff_minutes_median'] = final_df['created_at_diff_sec_median'] / 60
final_df.drop(0, axis=0, inplace=True)
final_df['created_at_diff_sec'] = final_df['created_at_diff_sec'].round(0)
final_df['created_at_diff_minutes'] = final_df['created_at_diff_minutes'].round(1)

final_df['created_at_diff_sec_median'] = final_df['created_at_diff_sec_median'].round(0)
final_df['created_at_diff_minutes_median'] = final_df['created_at_diff_minutes_median'].round(1)
final_df.reset_index(inplace=True)

In [29]:
final_df

,index,name_mop,created_at_diff_sec,created_at_diff_sec_median,created_at_diff_minutes,created_at_diff_minutes_median
0,1,Алина и Юля,268.0,200.0,4.5,3.3
1,2,Ангелина Милованова,386.0,243.0,6.4,4.0
2,3,Аня и Ксюша,772.0,546.0,12.9,9.1
3,4,Вика и Катя,831.0,572.0,13.9,9.5
4,5,Влада и Настя,283.0,183.0,4.7,3.0
5,6,Гюнель и Илина,510.0,308.0,8.5,5.1
6,7,Даша и Влада,1156.0,762.0,19.3,12.7
7,8,Даша и Даша,287.0,174.0,4.8,2.9
8,9,Даша и Карина,449.0,299.0,7.5,5.0
9,10,Ира и Варя,1533.0,911.0,25.5,15.2


In [138]:
# df.loc[(~df['created_at_diff_sec'].isna()) & (~df['not_work_time'] == True)].to_csv(
#     'el_first_dataset.csv', index=False, encoding="cp1251")
# final_df.to_csv('el_secons_dataset.csv', index=False, encoding="cp1251")

### Выводы по заданию 3:

Проанализарован период с 25.10 по 27.10 включительно.

Лучшее среднее время ответа на сообщения у Насти и Даши (#15).

Половина менеджеров имеют среднее время ответа до 8-ми минут и стараются придерживаться заднного темпа обработки.

Однако начиная с Ками и Милы (#11, 8+ мин на обработку) разброс между средним и медианой становится больше,
    что может говорить о том, что менеджер отвечает долго на некоторые сообщения или испытывает трудности с ответом.
    
Возможно менеджер Порхачева Полина(#19) имеет весомые затруднения при обработке сообщений (медиана увеличивается).

Худшие менеджеры по среднему времени обработки это Даша и Влада(#7) и Ира и Варя(#10), у которых не получатеся улучшить свой общий результат в отличие от худших Ксюша и Джамиля (#12), старающихся улучшить свой результат.
    

In [27]:
final_df.sort_values(by='created_at_diff_minutes')

,index,name_mop,created_at_diff_sec,created_at_diff_sec_median,created_at_diff_minutes,created_at_diff_minutes_median
14,15,Настя и Даша,196.0,104.0,3.3,1.7
0,1,Алина и Юля,268.0,200.0,4.5,3.3
4,5,Влада и Настя,283.0,183.0,4.7,3.0
7,8,Даша и Даша,287.0,174.0,4.8,2.9
13,14,Мария и Соня,312.0,155.0,5.2,2.6
16,17,Настя и Саша,339.0,316.0,5.7,5.3
12,13,Лиза и Ева,340.0,135.0,5.7,2.2
15,16,Настя и Малика,372.0,168.0,6.2,2.8
1,2,Ангелина Милованова,386.0,243.0,6.4,4.0
20,21,Софья Боднар,404.0,215.0,6.7,3.6
